# Action-Aware NanoWM — Day 3 model smoke test

This notebook freezes 32 VizDoom clips, verifies the frozen SD-VAE on real data, overfits NanoWM-S/2 for 500 steps, resumes to 600 steps in a fresh runtime, and writes the Day 3 evidence. Do not start the Week 2 development run until every final report says `passed`.

## 1. Runtime and persistent paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/NanoWMActionAware')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ChiCoTheLaAnh/NanoWMActionAware.git', str(REPO)], check=True)
os.chdir(REPO)
DATA_ROOT = Path('/content/drive/MyDrive/ActionAwareNanoWM/data')
PILOT_DIR = DATA_ROOT / 'vizdoom_basic' / 'pilot'
DAY3_DRIVE = Path('/content/drive/MyDrive/ActionAwareNanoWM/day3')
RUN1 = DAY3_DRIVE / 'run_initial_500'
RUN2 = DAY3_DRIVE / 'run_resumed_600'
EVIDENCE = DAY3_DRIVE / 'evidence'
SUBSET = EVIDENCE / 'fixed-train-clips.json'
for path in (RUN1, RUN2, EVIDENCE): path.mkdir(parents=True, exist_ok=True)
os.environ['DATASET_DIR'] = str(DATA_ROOT)
os.environ['RESULTS_DIR'] = str(DAY3_DRIVE / 'runs')
assert len(list(PILOT_DIR.glob('episode_*.hdf5'))) == 100, 'Expected the verified 100-episode pilot set'
print('Repository:', REPO)
print('Persistent Day 3 root:', DAY3_DRIVE)

In [ ]:
# Keep Colab's CUDA-compatible torch/torchvision stack.
%pip install -q -r requirements-colab.txt

## 2. Compose the exact smoke config and verify the real VizDoom VAE path

In [ ]:
import torch
assert torch.cuda.is_available(), 'Day 3 requires a GPU runtime'
overrides = [
    'experiment=vizdoom_smoke', 'dataset=game/vizdoom_basic', 'model=nanowm_s2',
    'wandb.enabled=false', 'logger.name=tensorboard',
    f'dataset_dir={DATA_ROOT}',
]
config_text = subprocess.check_output([sys.executable, 'src/main.py', '--cfg', 'job', *overrides], cwd=REPO, text=True)
RESOLVED = EVIDENCE / 'resolved-smoke-config.yaml'
RESOLVED.write_text(config_text)
subprocess.run([
    sys.executable, 'src/scripts/vizdoom_model_smoke.py', 'vae',
    '--config', str(RESOLVED), '--subset-manifest', str(SUBSET),
    '--output-dir', str(EVIDENCE), '--batch-size', '2',
], cwd=REPO, check=True)

## 3. Initial tiny-set overfit: steps 0–500

In [ ]:
initial_command = [
    sys.executable, 'src/main.py', *overrides,
    'experiment.training.max_steps=500',
    f'experiment.training.fixed_subset_path={SUBSET}',
    f'hydra.run.dir={RUN1}',
]
subprocess.run(initial_command, cwd=REPO, check=True)
before_candidates = sorted((RUN1 / 'checkpoints' / 'latest').glob('*.ckpt'))
assert before_candidates, 'Initial run did not write a checkpoint'
BEFORE = before_candidates[-1]
print('Resume checkpoint:', BEFORE)

## 4. Restart the runtime

Use **Runtime → Restart session**, then rerun Sections 1–2 only. Do not rerun the initial training cell. Continue below after `REPO`, `RUN1`, `RUN2`, `EVIDENCE`, `SUBSET`, `RESOLVED`, `overrides`, and `BEFORE` are restored. If needed, restore `BEFORE` with the first two lines in the next cell.

## 5. Resume full trainer state: steps 500–600

In [ ]:
before_candidates = sorted((RUN1 / 'checkpoints' / 'latest').glob('*.ckpt'))
BEFORE = before_candidates[-1]
resume_command = [
    sys.executable, 'src/main.py', *overrides,
    'experiment.training.max_steps=600',
    f'experiment.training.fixed_subset_path={SUBSET}',
    f'experiment.resume_from_checkpoint={BEFORE}',
    f'hydra.run.dir={RUN2}',
]
subprocess.run(resume_command, cwd=REPO, check=True)
after_candidates = sorted((RUN2 / 'checkpoints' / 'latest').glob('*.ckpt'))
assert after_candidates, 'Resumed run did not write a checkpoint'
AFTER = after_candidates[-1]
print('Post-resume checkpoint:', AFTER)

## 6. Produce loss, resume, motion, and comparison evidence

In [ ]:
subprocess.run([
    sys.executable, 'src/scripts/vizdoom_model_smoke.py', 'resume-report',
    '--before', str(BEFORE), '--after', str(AFTER),
    '--output', str(EVIDENCE / 'resume-report.json'), '--min-advanced-steps', '10',
], cwd=REPO, check=True)
logs = [path for path in (RUN1 / 'rank_0.log', RUN2 / 'rank_0.log') if path.exists()]
subprocess.run([
    sys.executable, 'src/scripts/vizdoom_model_smoke.py', 'loss-report',
    '--logs', *map(str, logs), '--output', str(EVIDENCE / 'training-report.json'), '--window', '10',
], cwd=REPO, check=True)
subprocess.run([
    sys.executable, 'src/scripts/vizdoom_model_smoke.py', 'render',
    '--config', str(RUN2 / 'config.yaml'), '--subset-manifest', str(SUBSET),
    '--checkpoint', str(AFTER), '--output-dir', str(EVIDENCE / 'predictions'),
    '--num-samples', '4', '--sampling-steps', '10',
], cwd=REPO, check=True)

## 7. Copy trackable evidence into the repository

In [ ]:
import json, shutil
TRACKED = REPO / 'reports' / 'evidence' / 'day3'
TRACKED.mkdir(parents=True, exist_ok=True)
shutil.copy2(RUN2 / 'config.yaml', EVIDENCE / 'resolved-smoke-config.yaml')
for name in ('fixed-train-clips.json', 'resolved-smoke-config.yaml', 'vae-report.json', 'vae-reconstruction.png', 'resume-report.json', 'training-report.json'):
    shutil.copy2(EVIDENCE / name, TRACKED / name)
prediction_report = EVIDENCE / 'predictions' / 'prediction-report.json'
shutil.copy2(prediction_report, TRACKED / prediction_report.name)
comparison = EVIDENCE / 'predictions' / 'sample_0000_compare.mp4'
shutil.copy2(comparison, TRACKED / comparison.name)
reports = [json.loads((TRACKED / name).read_text()) for name in ('vae-report.json', 'resume-report.json', 'training-report.json', 'prediction-report.json')]
assert all(report['status'] == 'passed' for report in reports), reports
print('Day 3 gate artifacts are ready for review at', TRACKED)
subprocess.run(['git', 'status', '--short'], cwd=REPO, check=True)